# ChEMBL 36 — Exploratory Data Analysis

Joined 10K sample from core ChEMBL 36 tables: activities, assays, targets, molecules, compound structures & properties.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt

DATA_PATH = "chembl_36/chembl_36_joined_10k.parquet"

df = pl.read_parquet(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 0. Filter to human targets only

## 1. Data overview

In [ ]:
# Check organism distribution before filtering
print("Top target organisms:")
print(df.group_by("target_organism").agg(pl.len().alias("count")).sort("count", descending=True).head(10))

n_before = len(df)
df = df.filter(pl.col("target_organism") == "Homo sapiens")
n_after = len(df)
print(f"\nFiltered to Homo sapiens only:")
print(f"  Before: {n_before:,}")
print(f"  After:  {n_after:,}")
print(f"  Removed: {n_before - n_after:,} non-human rows")

In [ ]:
df.describe()

In [ ]:
print(f"Columns ({len(df.columns)}):")
for col in df.columns:
    print(f"  {col:40s} {df[col].dtype}")

## 2. Deduplication by molecule

In [ ]:
n_before = len(df)
df = df.unique(subset=["molregno"], keep="last")
n_after = len(df)
print(f"Before: {n_before:,} rows")
print(f"After:  {n_after:,} rows")
print(f"Removed: {n_before - n_after:,} duplicates")

## 3. Null analysis

In [ ]:
nulls = df.null_count().to_dicts()[0]
sorted_nulls = {k: v for k, v in sorted(nulls.items(), key=lambda x: x[1], reverse=True) if v > 0}

print(f"Columns with nulls: {len(sorted_nulls)} / {len(df.columns)}\n")
for col, count in sorted_nulls.items():
    pct = count / len(df) * 100
    print(f"  {col:40s} {count:>6,} ({pct:5.1f}%)")

In [ ]:
columns = list(sorted_nulls.keys())
counts = list(sorted_nulls.values())

plt.figure(figsize=(14, 6))
plt.bar(columns, counts, color="salmon", edgecolor="black", alpha=0.7)
plt.axhline(y=len(df) * 0.5, color="red", linestyle="--", alpha=0.5, label="50% threshold")
plt.title(f"Missing values per column (total rows: {len(df):,})")
plt.xlabel("Column")
plt.ylabel("Null count")
plt.xticks(rotation=90, fontsize=8)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Unit checking

In [ ]:
unit_dist = (
    df.group_by("standard_units")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
print(f"Unique units: {len(unit_dist)}")
print(f"Rows with missing units: {df['standard_units'].null_count():,}\n")
unit_dist

## 5. Unit imputation

Fill missing `standard_units` with "nM" when `standard_value` exists and falls in 0.01–1,000,000 range.

In [ ]:
missing_before = df["standard_units"].null_count()

mask_missing = pl.col("standard_units").is_null() & pl.col("standard_value").is_not_null()
mask_range = (pl.col("standard_value") >= 0.01) & (pl.col("standard_value") <= 1e6)

df = df.with_columns([
    pl.when(mask_missing & mask_range)
    .then(pl.lit("nM"))
    .otherwise(pl.col("standard_units"))
    .alias("standard_units"),

    pl.when(mask_missing & mask_range)
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias("units_imputed"),
])

missing_after = df["standard_units"].null_count()
imputed = df.filter(pl.col("units_imputed")).height

print(f"Missing units before: {missing_before:,}")
print(f"Missing units after:  {missing_after:,}")
print(f"Imputed as nM:        {imputed:,}")

## 6. Unit standardization

Convert all concentration units to nM. Drop rows with non-convertible units (%, ug.mL-1, etc.).

In [ ]:
convertible_units = ["nM", "uM", "mM", "pM", "fM", "M"]
n_before = len(df)

# Drop rows with non-convertible units (%, ug.mL-1, etc.)
df = df.filter(pl.col("standard_units").is_in(convertible_units))

# Convert all values to nM in a new column
df = df.with_columns(
    pl.when(pl.col("standard_units") == "uM").then(pl.col("standard_value") * 1_000)
    .when(pl.col("standard_units") == "mM").then(pl.col("standard_value") * 1_000_000)
    .when(pl.col("standard_units") == "pM").then(pl.col("standard_value") * 0.001)
    .when(pl.col("standard_units") == "fM").then(pl.col("standard_value") * 0.000001)
    .when(pl.col("standard_units") == "M").then(pl.col("standard_value") * 1e9)
    .otherwise(pl.col("standard_value"))
    .alias("standard_value_nM"),
)

n_after = len(df)
print(f"Rows before: {n_before:,}")
print(f"Rows after:  {n_after:,}")
print(f"Dropped (non-convertible units): {n_before - n_after:,}")
print(f"\nOriginal unit breakdown (all now converted to nM):")
print(df.group_by("standard_units").agg(pl.len().alias("count")).sort("count", descending=True))
print(f"\nstandard_value_nM stats:")
print(df["standard_value_nM"].describe())

## 7. Outlier detection

Use log10 scale + IQR method to detect outliers in `standard_value_nM`. Cross-reference with ChEMBL's `data_validity_comment` flag.

In [ ]:
import numpy as np

# Work on log10 scale (bioactivity data is log-normal)
df_valid = df.filter(pl.col("standard_value_nM").is_not_null() & (pl.col("standard_value_nM") > 0))

log_values = np.log10(df_valid["standard_value_nM"].to_numpy())
q1, q3 = np.percentile(log_values, [25, 75])
iqr = q3 - q1
lower_bound = 10 ** (q1 - 1.5 * iqr)
upper_bound = 10 ** (q3 + 1.5 * iqr)

print(f"IQR on log10 scale: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}")
print(f"Outlier bounds (nM): [{lower_bound:.4f}, {upper_bound:,.0f}]")

# Flag outliers
df = df.with_columns(
    ((pl.col("standard_value_nM") < lower_bound) | (pl.col("standard_value_nM") > upper_bound))
    .alias("is_outlier")
)

n_outliers = df.filter(pl.col("is_outlier")).height
print(f"\nOutliers detected: {n_outliers:,} / {len(df):,} ({n_outliers/len(df)*100:.1f}%)")

# Cross-reference with data_validity_comment
print("\ndata_validity_comment among outliers:")
print(
    df.filter(pl.col("is_outlier"))
    .group_by("data_validity_comment")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

print("\ndata_validity_comment among non-outliers:")
print(
    df.filter(~pl.col("is_outlier"))
    .group_by("data_validity_comment")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

In [ ]:
# Chart 1: Histogram on log10 scale with outlier bounds
vals = df.filter(pl.col("standard_value_nM").is_not_null() & (pl.col("standard_value_nM") > 0))
log_vals = np.log10(vals["standard_value_nM"].to_numpy())
outlier_flags = vals["is_outlier"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: histogram with IQR bounds
axes[0].hist(log_vals, bins=60, color="steelblue", edgecolor="black", alpha=0.7)
axes[0].axvline(np.log10(lower_bound), color="red", linestyle="--", lw=2, label=f"Lower: {lower_bound:.2f} nM")
axes[0].axvline(np.log10(upper_bound), color="red", linestyle="--", lw=2, label=f"Upper: {upper_bound:,.0f} nM")
axes[0].set_xlabel("log10(standard_value_nM)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of activity values (log10 nM)")
axes[0].legend(fontsize=9)

# Right: scatter — index vs log10 value, outliers in red
axes[1].scatter(
    np.where(~outlier_flags)[0], log_vals[~outlier_flags],
    s=3, alpha=0.4, color="steelblue", label="Normal"
)
axes[1].scatter(
    np.where(outlier_flags)[0], log_vals[outlier_flags],
    s=15, alpha=0.8, color="red", label="Outlier", zorder=5
)
axes[1].axhline(np.log10(lower_bound), color="red", linestyle="--", lw=1, alpha=0.5)
axes[1].axhline(np.log10(upper_bound), color="red", linestyle="--", lw=1, alpha=0.5)
axes[1].set_xlabel("Row index")
axes[1].set_ylabel("log10(standard_value_nM)")
axes[1].set_title("Outliers highlighted (red)")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Remove outliers
n_before = len(df)
df = df.filter(~pl.col("is_outlier"))
n_after = len(df)
print(f"Removed {n_before - n_after:,} outliers")
print(f"Rows remaining: {n_after:,}")

## 8. Compute pIC50

`pIC50 = -log10(value_nM * 1e-9) = 9 - log10(value_nM)`

Compute new column, compare with ChEMBL's `pchembl_value`, then drop the original.

In [ ]:
# Compute pIC50 from standard_value_nM
df = df.with_columns(
    pl.when(pl.col("standard_value_nM").is_not_null() & (pl.col("standard_value_nM") > 0))
    .then(-(pl.col("standard_value_nM") * 1e-9).log(base=10))
    .otherwise(None)
    .alias("pIC50")
)

print(f"pIC50 computed: {df['pIC50'].drop_nulls().len():,} / {len(df):,} rows")
print(f"\npIC50 stats:")
print(df["pIC50"].describe())

In [ ]:
# Compare computed pIC50 with ChEMBL's pchembl_value
df_cmp = df.filter(pl.col("pIC50").is_not_null() & pl.col("pchembl_value").is_not_null())

df_cmp = df_cmp.with_columns(
    (pl.col("pIC50") - pl.col("pchembl_value")).abs().alias("abs_diff")
)

corr = df_cmp.select(pl.corr("pIC50", "pchembl_value")).item()
mean_diff = df_cmp["abs_diff"].mean()
median_diff = df_cmp["abs_diff"].median()
max_diff = df_cmp["abs_diff"].max()

print(f"Comparable rows: {len(df_cmp):,}")
print(f"Pearson correlation: {corr:.4f}")
print(f"Mean abs difference: {mean_diff:.4f}")
print(f"Median abs difference: {median_diff:.4f}")
print(f"Max abs difference: {max_diff:.4f}")

# Preview side by side
df_cmp.select(["molregno", "standard_value_nM", "pIC50", "pchembl_value", "abs_diff"]).head(10)

In [ ]:
# Drop original pchembl_value, keep our computed pIC50
df = df.drop("pchembl_value")
print(f"Dropped 'pchembl_value'. Keeping computed 'pIC50'.")
print(f"Final shape: {df.shape}")
df.select(["molregno", "standard_value_nM", "pIC50"]).head(10)

In [ ]:
pIC50_vals = df["pIC50"].drop_nulls().to_numpy()

plt.figure(figsize=(10, 5))
plt.hist(pIC50_vals, bins=50, color="steelblue", edgecolor="black", alpha=0.7)
plt.axvline(np.mean(pIC50_vals), color="red", linestyle="--", lw=1.5, label=f"Mean: {np.mean(pIC50_vals):.2f}")
plt.axvline(np.median(pIC50_vals), color="orange", linestyle="--", lw=1.5, label=f"Median: {np.median(pIC50_vals):.2f}")
plt.xlabel("pIC50")
plt.ylabel("Count")
plt.title("Distribution of computed pIC50")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Correlation analysis

Pearson correlation heatmap for all numeric columns. Identify highly correlated pairs (|r| > 0.9) that may be redundant.

In [ ]:
import seaborn as sns
import pandas as pd

# Select numeric columns only
numeric_cols = [c for c in df.columns if df[c].dtype in (pl.Int8, pl.Int16, pl.Int32, pl.Int64,
                                                          pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
                                                          pl.Float32, pl.Float64)]

print(f"Numeric columns: {len(numeric_cols)}")

# Compute correlation matrix via pandas (handles nulls gracefully)
corr_pd = df.select(numeric_cols).to_pandas().corr()

# Heatmap
plt.figure(figsize=(18, 15))
sns.heatmap(corr_pd, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.3, annot_kws={"size": 6},
            cbar_kws={"shrink": 0.8})
plt.title("Pearson correlation matrix (all numeric columns)")
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# Find highly correlated pairs (|r| > 0.8)
THRESHOLD = 0.8

pairs = []
for i in range(len(corr_pd.columns)):
    for j in range(i + 1, len(corr_pd.columns)):
        r = corr_pd.iloc[i, j]
        if abs(r) > THRESHOLD:
            pairs.append((corr_pd.columns[i], corr_pd.columns[j], round(r, 3)))

pairs.sort(key=lambda x: abs(x[2]), reverse=True)

print(f"Highly correlated pairs (|r| > {THRESHOLD}):\n")
for col1, col2, r in pairs:
    print(f"  {r:+.3f}  {col1}  <->  {col2}")

## 10. Drop redundant columns

Drop highly correlated, ID-only, and redundant columns. Keep what's needed for modeling.

In [ ]:
cols_before = len(df.columns)

# Highly correlated / redundant
DROP_CORRELATED = [
    "full_mwt",          # ~1.0 corr with mw_freebase
    "heavy_atoms",       # ~0.95 corr with mw_freebase
    "num_ro5_violations", # derived from hba/hbd/mw/alogp
    "standard_value",    # replaced by standard_value_nM
    "value",             # raw duplicate of standard_value
    "standard_value_nM", # replaced by pIC50
    "np_likeness_score", # niche, rarely useful
]

# ID / foreign key columns (no modeling value)
DROP_IDS = [
    "assay_id", "doc_id", "record_id", "assay_doc_id", "assay_src_id",
    "src_id", "tid", "cell_id", "tissue_id", "variant_id", "toid",
    "activity_id", "src_assay_id", "targcomp_id",
]

# Metadata / flags no longer needed
DROP_META = [
    "standard_flag", "potential_duplicate", "data_validity_comment",
    "is_outlier", "units_imputed",
    "bao_endpoint", "uo_units", "qudt_units", "bao_format",
    "standard_upper_value", "upper_value",
    "standard_relation", "relation", "type",
    "standard_units", "units", "standard_text_value", "text_value",
    "activity_comment", "action_type",
    "assay_test_type", "assay_category", "assay_strain",
    "assay_subcellular_fraction", "assay_tissue", "assay_cell_type",
    "assay_group", "aidx", "curated_by", "relationship_type",
    "confidence_score", "assay_chembl_id",
    "target_tax_id", "assay_tax_id", "species_group_flag",
    "standard_inchi",  # keep standard_inchi_key instead
    "full_molformula",
    # molecule metadata not needed for modeling
    "dosed_ingredient", "structure_type", "first_approval",
    "oral", "parenteral", "topical", "black_box_warning",
    "natural_product", "first_in_class", "chirality", "prodrug",
    "inorganic_flag", "usan_year", "availability_type",
    "usan_stem", "polymer_flag", "usan_substem", "usan_stem_definition",
    "withdrawn_flag", "chemical_probe", "orphan", "veterinary",
    "molecule_name", "molecule_chembl_id",
    "target_organism",  # already filtered to Homo sapiens
    "assay_organism",   # assay-level organism
    "description",      # assay description text
    "assay_type",       # assay metadata
    "max_phase", "therapeutic_flag",
    "ro3_pass",
]

to_drop = [c for c in DROP_CORRELATED + DROP_IDS + DROP_META if c in df.columns]
df = df.drop(to_drop)

cols_after = len(df.columns)
print(f"Dropped {cols_before - cols_after} columns ({cols_before} -> {cols_after})")
print(f"\nRemaining columns ({cols_after}):")
for col in df.columns:
    print(f"  {col:40s} {df[col].dtype}")